# Digit Recognizer - MNIST Competition

This notebook implements a CNN for digit recognition using the MNIST dataset from Kaggle's Digit Recognizer competition.

**Goal**: Achieve high accuracy on handwritten digit classification (0-9)


## Data Loading & Preprocessing

**Loads CSV data and creates typed train/test split function:**
- Loads training data (42,000 samples) and test data (28,000 samples)
- Creates `train_test_split_df()` wrapper with type safety
- Splits training data: 70% train, 30% validation
- Prepares data for CNN input format


In [ ]:
import keras
from keras import layers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from typing import Tuple


def train_test_split_df(
    X: pd.DataFrame, y: pd.Series, test_size: float = 0.3, random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Typed wrapper for train_test_split that ensures DataFrame/Series output types."""
    # Validate input types
    assert isinstance(X, pd.DataFrame)
    assert isinstance(y, pd.Series)

    # Perform the split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Ensure output types are correct
    X_train = (
        pd.DataFrame(X_train) if not isinstance(X_train, pd.DataFrame) else X_train
    )
    X_test = pd.DataFrame(X_test) if not isinstance(X_test, pd.DataFrame) else X_test
    y_train = pd.Series(y_train) if not isinstance(y_train, pd.Series) else y_train
    y_test = pd.Series(y_test) if not isinstance(y_test, pd.Series) else y_test

    return X_train, X_test, y_train, y_test


# Load the dataset
data_train: pd.DataFrame = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
data_train_y: pd.Series = data_train.pop("label")
data_train_x: pd.DataFrame = data_train

# Load the validation set
val_test: pd.DataFrame = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")
val_test_x: pd.DataFrame = val_test
val_test_y: pd.Series = pd.Series(np.zeros(val_test_x.shape[0]))

# Split the data using our typed wrapper
x_train, x_test, y_train, y_test = train_test_split_df(
    data_train_x, data_train_y, test_size=0.3, random_state=42
)

print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")
print(val_test_x.shape[0], "validation samples")

## Data Preparation

**Normalizes and reshapes data for CNN training:**
- Normalizes pixel values from [0,255] to [0,1]
- Reshapes 784-pixel vectors → 28×28×1 images
- Converts labels to categorical (one-hot) encoding
- Prepares training, validation, and test datasets

In [ ]:
num_classes = 10

# Normalize pixel values to [0,1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
val_test_x = val_test_x.astype("float32") / 255.0

# Reshape from (samples, 784) to (samples, 28, 28, 1)
x_train = x_train.values.reshape(-1, 28, 28, 1)
x_test = x_test.values.reshape(-1, 28, 28, 1)
val_test_x = val_test_x.values.reshape(-1, 28, 28, 1)

input_shape = (28, 28, 1)

# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

## Model Architecture

**Builds CNN with 3 convolutional blocks + dense layers:**
- Conv2D layers: 32 → 64 → 128 filters
- MaxPooling + Dropout for regularization
- Dense layers: 128 → 64 → 10 (output)
- Batch normalization for training stability
- ~250K parameters total

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.2),
        layers.Conv2D(128, kernel_size=(3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation="softmax"),
    ]
)
print(model.summary())

## Model Training

**Trains CNN with early stopping:**
- Compiles with categorical crossentropy loss + Adam optimizer
- Trains for 25 epochs with batch size 1024
- Early stopping: stops if val_loss doesn't improve for 5 epochs
- Restores best weights automatically
- Evaluates final test accuracy

In [ ]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

history = model.fit(
    x_train,
    y_train,
    epochs=25,
    batch_size=1024,
    validation_data=(x_test, y_test),
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5,
            monitor="val_loss",
            min_delta=0.01,  # type: ignore[arg-type]
            restore_best_weights=True,
        )
    ],
)

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

## Training Visualization

**Plots training progress:**
- Shows training vs validation accuracy over epochs
- Y-axis limited to [0.9, 1.0] for better visualization
- Helps identify overfitting and convergence patterns


In [ ]:
plt.plot(history.history["accuracy"], label="accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.ylim([0.9, 1])
plt.legend(loc="lower right")
plt.show()

## Kaggle Submission

**Generates predictions for Kaggle submission:**
- Makes predictions on test set (28,000 samples)
- Converts probabilities to digit labels
- Creates submission.csv with ImageId and Label columns
- Ready for Kaggle upload


In [ ]:
# Now make predictions on the validation set
val_predictions = model.predict(val_test_x)
val_predictions = np.argmax(val_predictions, axis=1)

# Create a submission file
submission = pd.DataFrame(
    {"ImageId": range(1, len(val_predictions) + 1), "Label": val_predictions}
)
submission.to_csv("/kaggle/working/submission.csv", index=False)

## Model Testing & Error Analysis

**Independent testing on MNIST dataset:**
- Combines MNIST train + test sets (70,000 samples)
- Randomly samples 10,000 examples for testing
- Calculates accuracy and error statistics
- Done to check for overfitting on the training set
- **Error Analysis:**
  - Bar chart showing errors by digit
  - 4×3 grid of misclassified examples
  - Error rate and digit-specific breakdowns


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct

# ---------- Load test images ----------
with open("/kaggle/input/mnist-dataset/t10k-images.idx3-ubyte", "rb") as f:
    _, num, rows, cols = struct.unpack(">IIII", f.read(16))
    x_test = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)

# ---------- Load test labels ----------
with open("/kaggle/input/mnist-dataset/t10k-labels.idx1-ubyte", "rb") as f:
    _, num = struct.unpack(">II", f.read(8))
    y_test = np.frombuffer(f.read(), dtype=np.uint8)

# ---------- Preprocess ----------
x_test = x_test.astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1)

# ---------- Predict ----------
print("Running inference on test set...")
predictions = model.predict(x_test, verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

# ---------- Accuracy ----------
correct = np.sum(predicted_labels == y_test)
accuracy = correct / len(y_test)
print(f"🎯 Results: {correct}/{len(y_test)} correct ({accuracy:.4f} = {accuracy*100:.2f}%)")

# ---------- Error analysis ----------
misclassified = predicted_labels != y_test
error_count = np.sum(misclassified)
print(f"Errors: {error_count}")
print(f"Error rate: {error_count/len(y_test)*100:.2f}%")

error_by_digit = [np.sum(y_test[misclassified] == d) for d in range(10)]
print(f"Errors by digit: {dict(enumerate(error_by_digit))}")

plt.figure(figsize=(10, 4))
plt.bar(range(10), error_by_digit, alpha=0.7)
plt.xlabel("True Digit")
plt.ylabel("Errors")
plt.title("Errors by Digit")
plt.xticks(range(10))
plt.show()

# ---------- Misclassified examples ----------
error_indices = np.where(misclassified)[0][:12]
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
fig.suptitle("Misclassified Examples (True → Predicted)", fontsize=14)

for i, idx in enumerate(error_indices):
    r, c = divmod(i, 4)
    axes[r, c].imshow(x_test[idx].reshape(28, 28), cmap="gray")
    axes[r, c].set_title(f"{y_test[idx]}→{predicted_labels[idx]}", color="red")
    axes[r, c].axis("off")

for i in range(len(error_indices), 12):
    r, c = divmod(i, 4)
    axes[r, c].axis("off")

plt.tight_layout()
plt.show()
